# E4 · Inspect frames where the losses agree or disagree (Endoscapes)

Reads `results/phase2/endoscapes/e4_frames.csv` (written by `python -m b2j.e4 endoscapes`; one row per test frame and
criterion) and the local images in `../datasets/endoscapes`.

Each loss is one model: the mean of its 5 seeds' scores (hard / soft BCE: p; B²: B1 = P(at least 2 of 3 raters say yes)),
with a cut-off fitted on validation. A call is right when it matches the majority of the 3 raters.

`combo` is one of: `all right`, `only <loss> wrong`, `only <loss> right`, `all wrong`
(losses: `hard_bce`, `soft_bce`, `b2_b1`). `exp`: `E4a` = best-BACC checkpoint, `E4b` = own-loss checkpoint.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from b2j.data import load_table

frames = pd.read_csv(REPO / "results" / "phase2" / "endoscapes" / "e4_frames.csv")
table = load_table("endoscapes")
lab_idx = {"C1": 0, "C2": 1, "C3": 2}
paths = dict(zip(table.ids, table.paths))
votes = {i: v for i, v in zip(table.ids, table.votes)}          # [3 raters, 3 criteria]
frames["path"] = frames["id"].map(paths)
assert frames["path"].notna().all()
MODELS = {"hard_bce": "Hard BCE", "soft_bce": "Soft BCE", "b2_b1": "B²-B1"}
COMBOS = ["all right"] + [f"only {m} wrong" for m in MODELS] + [f"only {m} right" for m in MODELS] + ["all wrong"]
print(len(frames), "rows;", frames["id"].nunique(), "test frames")

## Counts: right/wrong combination × vote pattern

In [ ]:
def counts(exp="E4a", label="C1"):
    sub = frames[(frames.exp == exp) & (frames.label == label)]
    t = pd.crosstab(sub.combo, sub.pattern).reindex(COMBOS).fillna(0).astype(int)
    t = t[sorted(t.columns, key=lambda s: int(s.split("/")[0]))]
    t["all"] = t.sum(1)
    return t

for lab in ("C1", "C2", "C3"):
    print(f"\n{lab}  (E4a)")
    display(counts("E4a", lab))

## Show frames

`show(exp, label, combo=None, pattern=None, n=12, seed=0, order="random", crop=True)`

- `combo` / `pattern` filter the frames (e.g. `combo="only b2_b1 right"`, `pattern="2/3"`); `None` = any.
- `order`: `"random"` (reproducible with `seed`), or a loss name to sort by that loss's distance from its cut-off
  (closest first).
- `crop=True` shows the centre 480 × 480 crop the models saw (before resizing to 256); `False` the full frame.

Caption: frame id, the three raters' votes for this criterion, majority, then each loss's score vs its cut-off,
its call (yes / no) and ✓ / ✗.

In [ ]:
def _caption(r):
    v = votes[r.id][:, lab_idx[r.label]].astype(int)
    lines = [f"{r.id}  {r.label}  raters {v.tolist()}  ({r.pattern}, majority {'yes' if r.majority else 'no'})"]
    for m, name in MODELS.items():
        mark = "✓" if r[f"{m}_right"] else "✗"
        call = "yes" if r[f"{m}_call"] else "no"
        lines.append(f"{name}: {r[f'{m}_score']:.3f} vs {r[f'{m}_cutoff']:.3f} → {call} {mark}")
    return "\n".join(lines)


def _load(path, crop):
    im = Image.open(path).convert("RGB")
    if crop:
        w, h = im.size
        s = min(480, w, h)
        im = im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return im


def show(exp="E4a", label="C1", combo=None, pattern=None, n=12, seed=0, order="random", crop=True, cols=4):
    sub = frames[(frames.exp == exp) & (frames.label == label)]
    if combo is not None:
        sub = sub[sub.combo == combo]
    if pattern is not None:
        sub = sub[sub.pattern == pattern]
    print(f"{len(sub)} frames match; showing {min(n, len(sub))}")
    if sub.empty:
        return sub
    if order == "random":
        sub = sub.sample(min(n, len(sub)), random_state=seed)
    else:
        d = (np.log(sub[f"{order}_score"] / (1 - sub[f"{order}_score"]))
             - np.log(sub[f"{order}_cutoff"] / (1 - sub[f"{order}_cutoff"]))).abs()
        sub = sub.assign(_d=d).sort_values("_d").head(n)
    rows = int(np.ceil(len(sub) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 5.4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        ax.imshow(_load(r.path, crop))
        ax.set_title(_caption(r), fontsize=7.5, loc="left", family="monospace")
    plt.tight_layout()
    plt.show()
    return sub.drop(columns=["path"])

### Examples

In [ ]:
# Frames on C2 where only B² gets the majority right
show("E4a", "C2", combo="only b2_b1 right");

In [ ]:
# Frames on C1 where only B² is wrong
show("E4a", "C1", combo="only b2_b1 wrong");

In [ ]:
# Split frames (2 of 3 raters say yes) on C3 that every loss gets wrong
show("E4a", "C3", combo="all wrong", pattern="2/3");

In [ ]:
# Unanimous yes (3/3) on C2, any combination, the ones B² is least sure about first
show("E4a", "C2", pattern="3/3", order="b2_b1");

## Interactive (needs ipywidgets)

In [ ]:
try:
    import ipywidgets as w
    w.interact(lambda exp, label, combo, pattern, n, seed, crop: show(exp, label, None if combo == "any" else combo,
                                                                       None if pattern == "any" else pattern, n, seed,
                                                                       crop=crop) and None,
               exp=["E4a", "E4b"], label=["C1", "C2", "C3"], combo=["any"] + COMBOS,
               pattern=["any", "0/3", "1/3", "2/3", "3/3"], n=w.IntSlider(12, 4, 40, 4), seed=w.IntText(0),
               crop=True)
except ImportError:
    print("ipywidgets not installed: use show(...) above")